# Homework 8: Predicting Shortwave Radiation — Model Comparison & SHAP

**F&W ECOL 458 — Environmental Data Science**  
**[YOUR NAME]:**
**Submission:** Upload your completed `.ipynb` to GitHub and submit the link to Canvas. **Before submitting, do Runtime > Restart and run all** to ensure all outputs are current.

---

### Overview

In this assignment you will predict incoming shortwave radiation at the Earth's surface from atmospheric and surface variables. You will train **all regression algorithms covered in the course**, compare their performance, and use **SHAP** to interpret the Random Forest and ANN models.

**Point breakdown:**

| Part | Topic | Points |
|---|---|---|
| 1 | Model comparison — all regressors | 55 |
| 2 | SHAP interpretation — Random Forest & ANN | 45 |
| **Total** | | **100** |

### The dataset

| Feature | Description | Unit |
|---|---|---|
| SZA | Solar Zenith Angle | degrees |
| AOD | Aerosol Optical Depth | unitless |
| COD | Cloud Optical Depth | unitless |
| CLD_FRAC | Cloud Fraction | 0-1 |
| UW | Water Column | cm |
| TO3 | Total Ozone | DU |
| Pressure | Surface Air Pressure | hPa |
| BSA | Black-Sky Albedo | 0-1 |
| WSA | White-Sky Albedo | 0-1 |

Targets: **SW_direct** (direct beam), **SW_diffuse** (scattered), and **SW = SW_direct + SW_diffuse** (total).

### Setup


In [ ]:
# Run this cell first
!pip install xgboost shap -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')
print("Setup complete.")


In [ ]:
# ── Load data ──
# Replace with: df = pd.read_csv('SW.csv') if the file is available
np.random.seed(42)
n = 5000
SZA = np.random.uniform(0, 85, n)
AOD = np.random.exponential(0.15, n)
COD = np.random.exponential(5, n)
CLD_FRAC = np.random.uniform(0, 1, n)
UW = np.random.uniform(0.5, 5, n)
TO3 = np.random.uniform(250, 450, n)
Pressure = np.random.uniform(850, 1050, n)
BSA = np.random.uniform(0.05, 0.4, n)
WSA = np.random.uniform(0.05, 0.4, n)

cos_sza = np.cos(np.radians(SZA))
SW_direct = np.maximum(0, 1361 * cos_sza * np.exp(-AOD / np.clip(cos_sza, 0.05, 1))
                       * (1 - CLD_FRAC * 0.9) + np.random.normal(0, 15, n))
SW_diffuse = np.maximum(0, 200 * cos_sza * (0.3 + 0.5 * CLD_FRAC + 0.3 * AOD)
                        + np.random.normal(0, 10, n))

df = pd.DataFrame({
    'SZA': SZA, 'AOD': AOD, 'COD': COD, 'CLD_FRAC': CLD_FRAC,
    'UW': UW, 'TO3': TO3, 'Pressure': Pressure, 'BSA': BSA, 'WSA': WSA,
    'SW_direct': SW_direct, 'SW_diffuse': SW_diffuse
})
df['SW'] = df['SW_direct'] + df['SW_diffuse']

feature_cols = ['SZA', 'AOD', 'COD', 'CLD_FRAC', 'UW', 'TO3', 'Pressure', 'BSA', 'WSA']
print(f"Dataset: {df.shape[0]} samples, {len(feature_cols)} features, 3 targets")


## Part 1: Model Comparison — All Regressors (55 points)


### 1.1 Prepare train/test split (5 pts)

1. Define `X` (9 features), `y_total` (SW), `y_direct` (SW_direct), and `y_diffuse` (SW_diffuse).
2. Split into 75% train / 25% test (`random_state=42`).
3. Create **scaled** versions using `StandardScaler` for algorithms that require it.


In [ ]:
# YOUR CODE HERE




### 1.2 Train all regressors (20 pts)

Train the following **7 regressors** on `SW` (total shortwave):

| Algorithm | Use scaled data? | Key parameters |
|---|---|---|
| Linear Regression | Yes | default |
| KNN (k=10) | Yes | `n_neighbors=10` |
| SVR (RBF) | Yes | `C=100` |
| Decision Tree | No | `max_depth=10` |
| Random Forest | No | `n_estimators=200` |
| XGBoost | No | `n_estimators=500, max_depth=6, learning_rate=0.1` |
| ANN (MLP) | Yes | `MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, early_stopping=True)` |

For each, record **test R$^2$**, **test RMSE**, and **training time**.

Print a summary table and create:
1. A **bar chart** comparing test R$^2$ across all 7 models.
2. A **2x4 grid of 1:1 scatter plots** (observed vs. predicted) with R$^2$ in each title. Leave the 8th subplot empty.

**Answer:** Which model achieves the best test R$^2$? Which is fastest? Why do tree-based ensembles outperform linear regression on this dataset?


In [ ]:
# YOUR CODE HERE




*Your written answer here:*



### 1.3 Predict all three radiation components (15 pts)

Using **Random Forest** and **ANN (MLP)** separately:

1. Train each on **SW_direct**, **SW_diffuse**, and **SW_total** (3 targets x 2 models = 6 models total).
2. Create a **2x3 grid of 1:1 plots**: top row = Random Forest, bottom row = ANN. Columns = SW_direct, SW_diffuse, SW_total. Show R$^2$ and RMSE in each title.

**Answer:**
1. Which radiation component is hardest to predict? Why might this be the case physically?
2. Does Random Forest or ANN perform better overall? Is the difference consistent across all three components?


In [ ]:
# YOUR CODE HERE




*Your written answer here:*



### 1.4 Feature importance (15 pts)

1. Extract **Gini feature importance** from the Random Forest model (SW_total).
2. Compute **permutation importance** for the same model (`n_repeats=10` on the test set).
3. Plot both as side-by-side horizontal bar charts.


In [ ]:
# YOUR CODE HERE




## Part 2: SHAP Interpretation — Random Forest & ANN (45 points)

Use `shap.TreeExplainer` for Random Forest and `shap.KernelExplainer` for ANN.

For `KernelExplainer`, use a subsample to keep computation manageable:
```python
background = shap.sample(X_train_sc, 100)
explainer_ann = shap.KernelExplainer(mlp_model.predict, background)
shap_values_ann = explainer_ann.shap_values(X_test_sc[:200])
```


### 2.1 SHAP for Random Forest (20 pts)

1. Compute SHAP values for your Random Forest (SW_total) on the test set.
2. Produce a **SHAP beeswarm (summary) plot**.
3. Produce a **SHAP bar plot** (mean |SHAP value| per feature).
4. Produce **SHAP dependence plots** for **SZA**, **CLD_FRAC**, and **AOD**.

**Answer:**
1. In the beeswarm plot, does high SZA increase or decrease SW? Is this physically correct?
2. What is the direction of effect for CLD_FRAC? Explain the physics in one sentence.
3. In the SZA dependence plot, is the relationship linear or nonlinear? Describe its shape and explain why.
4. In the AOD dependence plot, what does the interaction color reveal?


In [ ]:
# YOUR CODE HERE




*Your written answers here:*



### 2.2 SHAP for ANN — Comparison with Random Forest (25 pts)

1. Compute SHAP values for your ANN (SW_total) using `KernelExplainer` on 200 test samples.
2. Display the **RF beeswarm** and **ANN beeswarm** side by side.
3. Display the **RF bar plot** and **ANN bar plot** side by side.

**Answer:**
1. Do RF and ANN agree on the top 3 most important features? List them for each model.
2. Are the directions of effect the same for both models?
3. If there are any differences, explain why a tree-based model and a neural network might learn slightly different representations of the same physics.
4. Based on both accuracy (Part 1) and SHAP (this section), which model would you trust more for this task and why?


In [ ]:
# YOUR CODE HERE




*Your written answers here:*

